
# Predicción del **next close** con **PySpark** en Google Colab

Este notebook carga un CSV OHLCV como el adjunto (`kucoin__2022__ETH-USDT_15min.csv`) y entrena un modelo de **regresión** para predecir el **`Close` de la vela siguiente**.

## Objetivo
Construir un pipeline de ML en **PySpark** que:

1. Cargue el dataset en Spark.
2. Ordene la serie temporal correctamente.
3. Cree la variable objetivo `next_close`.
4. Genere *features* de precio, volumen, *lags* y medias móviles.
5. Haga un **split temporal** train/test.
6. Entrene un modelo de regresión.
7. Evalúe el resultado con **RMSE**, **MAE** y **R²**.

> Importante: el notebook evita fuga de información usando solo datos de la vela actual y anteriores para predecir la siguiente.


In [ ]:

# Instalar PySpark en Colab
!pip -q install pyspark


In [ ]:

# Opción recomendada en Colab: subir el CSV manualmente
from google.colab import files

uploaded = files.upload()
CSV_FILE = f"/content/{next(iter(uploaded.keys()))}"

print("Archivo cargado:", CSV_FILE)

# Si prefieres usar Google Drive, comenta las 4 líneas anteriores y usa algo como:
# CSV_FILE = "/content/drive/MyDrive/tu_ruta/kucoin__2022__ETH-USDT_15min.csv"


Saving kucoin__v2020_nov_ETH-USDT_1hour.csv to kucoin__v2020_nov_ETH-USDT_1hour (1).csv
Archivo cargado: /content/kucoin__v2020_nov_ETH-USDT_1hour (1).csv


In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("next-close-regression")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark



## 1. Carga del dataset
El CSV adjunto tiene estas columnas:

- `Time`
- `time`
- `Open`
- `Close`
- `High`
- `Low`
- `Amount`
- `Volume`
- `Datetime`

Usaremos `Datetime` y `time` para ordenar correctamente la serie.


In [ ]:

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_FILE)
)

print("Número de filas:", df.count())
print("Columnas:", df.columns)
df.show(5, truncate=False)


Número de filas: 51912
Columnas: ['Time0', 'time1', 'Open', 'Close', 'High', 'Low', 'Amount', 'Volume', 'Datetime']
+-------------------+----------+------+------+------+------+------------+----------------+-------------------+
|Time0              |time1     |Open  |Close |High  |Low   |Amount      |Volume          |Datetime           |
+-------------------+----------+------+------+------+------+------------+----------------+-------------------+
|2019-12-31 23:00:00|1577833200|129.12|129.22|129.42|128.96|362.73128201|46855.6222082959|2019-12-31 23:00:00|
|2020-01-01 00:00:00|1577836800|129.22|128.9 |129.22|128.77|269.21894455|34706.9746788605|2020-01-01 00:00:00|
|2020-01-01 01:00:00|1577840400|128.9 |130.59|130.6 |128.88|482.74006982|62690.1867964104|2020-01-01 01:00:00|
|2020-01-01 02:00:00|1577844000|130.58|130.87|130.96|130.41|354.91194192|46397.5206118871|2020-01-01 02:00:00|
|2020-01-01 03:00:00|1577847600|130.87|130.16|130.93|129.97|735.02069587|95795.5795053984|2020-01-01 03:00:

In [ ]:

# Parseo y normalización de tipos
df = (
    df
    .withColumn("Datetime", F.to_timestamp("Datetime"))

    .withColumn("Open", F.col("Open").cast("double"))
    .withColumn("High", F.col("High").cast("double"))
    .withColumn("Low", F.col("Low").cast("double"))
    .withColumn("Close", F.col("Close").cast("double"))
    .withColumn("Amount", F.col("Amount").cast("double"))
    .withColumn("Volume", F.col("Volume").cast("double"))
    .orderBy("Datetime")
)

df.printSchema()
df.select("Datetime", "Open", "High", "Low", "Close", "Amount", "Volume").show(5, truncate=False)


root
 |-- Time0: timestamp (nullable = true)
 |-- time1: integer (nullable = true)
 |-- Open: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Datetime: timestamp (nullable = true)

+-------------------+------+------+------+------+------------+----------------+
|Datetime           |Open  |High  |Low   |Close |Amount      |Volume          |
+-------------------+------+------+------+------+------------+----------------+
|2019-12-31 23:00:00|129.12|129.42|128.96|129.22|362.73128201|46855.6222082959|
|2020-01-01 00:00:00|129.22|129.22|128.77|128.9 |269.21894455|34706.9746788605|
|2020-01-01 01:00:00|128.9 |130.6 |128.88|130.59|482.74006982|62690.1867964104|
|2020-01-01 02:00:00|130.58|130.96|130.41|130.87|354.91194192|46397.5206118871|
|2020-01-01 03:00:00|130.87|130.93|129.97|130.16|735.02069587|95795.5795053984|
+------------


## 2. Crear la variable objetivo: `next_close`

La etiqueta será el `Close` de la siguiente vela:

- fila actual → *features*
- fila siguiente → `next_close`


In [ ]:

w_order = Window.orderBy("Datetime")

df = df.withColumn("next_close", F.lead("Close", 1).over(w_order))
df.select("Datetime", "Close", "next_close").show(10, truncate=False)


+-------------------+------+----------+
|Datetime           |Close |next_close|
+-------------------+------+----------+
|2019-12-31 23:00:00|129.22|128.9     |
|2020-01-01 00:00:00|128.9 |130.59    |
|2020-01-01 01:00:00|130.59|130.87    |
|2020-01-01 02:00:00|130.87|130.16    |
|2020-01-01 03:00:00|130.16|130.22    |
|2020-01-01 04:00:00|130.22|130.29    |
|2020-01-01 05:00:00|130.29|130.42    |
|2020-01-01 06:00:00|130.42|130.27    |
|2020-01-01 07:00:00|130.27|130.34    |
|2020-01-01 08:00:00|130.34|130.16    |
+-------------------+------+----------+
only showing top 10 rows



## 3. Feature engineering

Se construyen *features* usando únicamente información disponible hasta la vela actual:

- OHLCV actual
- rangos y retornos
- lags de precio y volumen
- medias móviles
- volatilidad rolling
- variables de calendario


In [ ]:
# Ventanas
w_lag = Window.orderBy("Datetime")
w_ma_3 = Window.orderBy("Datetime").rowsBetween(-2, 0)
w_ma_6 = Window.orderBy("Datetime").rowsBetween(-5, 0)
w_ma_12 = Window.orderBy("Datetime").rowsBetween(-11, 0)

feat_df = (
    df
    # Variables temporales
    .withColumn("hour", F.hour("Datetime"))
    .withColumn("minute", F.minute("Datetime"))
    .withColumn("dayofweek", F.dayofweek("Datetime"))

    # Derivadas OHLC
    .withColumn("hl_range", F.col("High") - F.col("Low"))
    .withColumn("oc_change", F.col("Close") - F.col("Open"))
    .withColumn("co_ratio", F.when(F.col("Open") != 0, F.col("Close") / F.col("Open")).otherwise(None))
    .withColumn("hl_ratio", F.when(F.col("Low") != 0, F.col("High") / F.col("Low")).otherwise(None))

    # Lags de close (1,2,3,4)
    .withColumn("close_lag_1", F.lag("Close", 1).over(w_lag))
    .withColumn("close_lag_2", F.lag("Close", 2).over(w_lag))
    .withColumn("close_lag_3", F.lag("Close", 3).over(w_lag))
    .withColumn("close_lag_4", F.lag("Close", 4).over(w_lag))

    # Lags de volumen y amount (1,2 para ambas)
    .withColumn("volume_lag_1", F.lag("Volume", 1).over(w_lag))
    .withColumn("volume_lag_2", F.lag("Volume", 2).over(w_lag))
    .withColumn("amount_lag_1", F.lag("Amount", 1).over(w_lag))
    .withColumn("amount_lag_2", F.lag("Amount", 2).over(w_lag))

    # Retornos y cambios relativos
    .withColumn(
        "return_1",
        F.when(F.lag("Close", 1).over(w_lag) != 0,
               (F.col("Close") - F.lag("Close", 1).over(w_lag)) / F.lag("Close", 1).over(w_lag))
         .otherwise(None)
    )
    .withColumn(
        "volume_change_1",
        F.when(F.lag("Volume", 1).over(w_lag) != 0,
               (F.col("Volume") - F.lag("Volume", 1).over(w_lag)) / F.lag("Volume", 1).over(w_lag))
         .otherwise(None)
    )

    # Medias móviles (close y volume)
    .withColumn("sma_close_3", F.avg("Close").over(w_ma_3))
    .withColumn("sma_close_6", F.avg("Close").over(w_ma_6))
    .withColumn("sma_close_12", F.avg("Close").over(w_ma_12))
    .withColumn("sma_volume_6", F.avg("Volume").over(w_ma_6))

    # Volatilidad rolling
    .withColumn("std_close_6", F.stddev("Close").over(w_ma_6))
    .withColumn("std_close_12", F.stddev("Close").over(w_ma_12))
)

feat_df.select(
    "Datetime", "Close", "next_close", "close_lag_1", "return_1",
    "sma_close_3", "sma_close_6", "std_close_6"
).show(10, truncate=False)

+-------------------+------+----------+-----------+----------------------+------------------+------------------+-------------------+
|Datetime           |Close |next_close|close_lag_1|return_1              |sma_close_3       |sma_close_6       |std_close_6        |
+-------------------+------+----------+-----------+----------------------+------------------+------------------+-------------------+
|2019-12-31 23:00:00|129.22|128.9     |NULL       |NULL                  |129.22            |129.22            |NULL               |
|2020-01-01 00:00:00|128.9 |130.59    |129.22     |-0.002476396842593973 |129.06            |129.06            |0.22627416997969038|
|2020-01-01 01:00:00|130.59|130.87    |128.9      |0.013110938712179967  |129.57000000000002|129.57000000000002|0.8977193325310535 |
|2020-01-01 02:00:00|130.87|130.16    |130.59     |0.0021441151696148337 |130.12            |129.895           |0.9796768174590406 |
|2020-01-01 03:00:00|130.16|130.22    |130.87     |-0.005425231145411

In [ ]:

# Eliminar nulos generados por lags/medias y por la última fila sin target
model_df = feat_df.dropna()

print("Filas listas para modelado:", model_df.count())
model_df.select("Datetime", "Close", "next_close").show(5, truncate=False)


Filas listas para modelado: 51892
+-------------------+------+----------+
|Datetime           |Close |next_close|
+-------------------+------+----------+
|2020-01-01 03:00:00|130.16|130.22    |
|2020-01-01 04:00:00|130.22|130.29    |
|2020-01-01 05:00:00|130.29|130.42    |
|2020-01-01 06:00:00|130.42|130.27    |
|2020-01-01 07:00:00|130.27|130.34    |
+-------------------+------+----------+
only showing top 5 rows



## 4. Split temporal train/test

En series temporales **no** conviene mezclar observaciones aleatoriamente.  
Aquí usamos el 80% inicial para entrenamiento y el 20% final para test.


In [ ]:

# Añadimos row_id para hacer un corte temporal
model_df = model_df.withColumn("row_id", F.row_number().over(Window.orderBy("Datetime")))

n_rows = model_df.count()
train_size = int(n_rows * 0.80)

train_df = model_df.filter(F.col("row_id") <= train_size)
test_df = model_df.filter(F.col("row_id") > train_size)

print("Total:", n_rows)
print("Train:", train_df.count())
print("Test:", test_df.count())

train_df.select(F.min("Datetime"), F.max("Datetime")).show()
test_df.select(F.min("Datetime"), F.max("Datetime")).show()


Total: 51892
Train: 41513
Test: 10379
+-------------------+-------------------+
|      min(Datetime)|      max(Datetime)|
+-------------------+-------------------+
|2020-01-01 03:00:00|2024-09-26 10:00:00|
+-------------------+-------------------+

+-------------------+-------------------+
|      min(Datetime)|      max(Datetime)|
+-------------------+-------------------+
|2024-09-26 11:00:00|2025-12-02 21:00:00|
+-------------------+-------------------+




## 5. Preparar features y entrenar el modelo

Usaremos un **RandomForestRegressor** de PySpark.  
Es una buena opción como baseline potente para datos tabulares.

> Nota: en predicción financiera real, acertar el precio exacto suele ser difícil. El valor educativo aquí está en construir correctamente el pipeline temporal.


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

# 1. FORZAR LA CACHÉ (Esto es la clave para que no se quede "colgado")
train_df.cache()
test_df.cache()
print(f"Filas en train_df (caché materializada): {train_df.count()}")

feature_cols = [
    "Open", "High", "Low", "Close", "Amount", "Volume",
    "hour", "minute", "dayofweek",
    "hl_range", "oc_change", "co_ratio", "hl_ratio",
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4",
    "volume_lag_1", "volume_lag_2", "amount_lag_1", "amount_lag_2",
    "return_1", "volume_change_1",
    "sma_close_3", "sma_close_6", "sma_close_12", "sma_volume_6",
    "std_close_6", "std_close_12"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

# 2. MODELO UN POCO MÁS LIGERO (para que Colab no sufra)
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="next_close",
    predictionCol="prediction",
    numTrees=50,       # <-- Reducido de 150 a 50
    maxDepth=7,        # <-- Reducido de 10 a 7
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])

print("Entrenando el modelo, por favor espera un par de minutos...")
model    = pipeline.fit(train_df)
pred_df  = model.transform(test_df)

print("¡Modelo entrenado!")
pred_df.select("Datetime", "Close", "next_close", "prediction").show(10, truncate=False)

+-------------------+-------+----------+------------------+
|Datetime           |Close  |next_close|prediction        |
+-------------------+-------+----------+------------------+
|2024-09-26 11:00:00|2628.65|2630.2    |2666.37241532171  |
|2024-09-26 12:00:00|2630.2 |2620.63   |2666.8166570937465|
|2024-09-26 13:00:00|2620.63|2613.59   |2667.243489003602 |
|2024-09-26 14:00:00|2613.59|2652.15   |2669.6137896383525|
|2024-09-26 15:00:00|2652.15|2650.17   |2680.9092507194864|
|2024-09-26 16:00:00|2650.17|2629.8    |2671.6707220070543|
|2024-09-26 17:00:00|2629.8 |2659.56   |2670.875943836145 |
|2024-09-26 18:00:00|2659.56|2649.43   |2679.7181009967767|
|2024-09-26 19:00:00|2649.43|2631.55   |2669.3691840622682|
|2024-09-26 20:00:00|2631.55|2638.52   |2666.4044528002273|
+-------------------+-------+----------+------------------+
only showing top 10 rows



## 6. Evaluación del modelo
Evaluamos con:

- **RMSE**
- **MAE**
- **R²**

Además comparamos contra una baseline ingenua:

**predecir que el siguiente cierre será igual al cierre actual**.


In [ ]:

from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(labelCol="next_close", predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="next_close", predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="next_close", predictionCol="prediction", metricName="r2")

rmse = rmse_eval.evaluate(pred_df)
mae = mae_eval.evaluate(pred_df)
r2 = r2_eval.evaluate(pred_df)

print("=== Modelo RandomForest ===")
print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"R²  : {r2:.6f}")

# Baseline naive: next_close ≈ Close actual
baseline_df = test_df.withColumn("baseline_prediction", F.col("Close"))

baseline_rmse = RegressionEvaluator(
    labelCol="next_close", predictionCol="baseline_prediction", metricName="rmse"
).evaluate(baseline_df)

baseline_mae = RegressionEvaluator(
    labelCol="next_close", predictionCol="baseline_prediction", metricName="mae"
).evaluate(baseline_df)

baseline_r2 = RegressionEvaluator(
    labelCol="next_close", predictionCol="baseline_prediction", metricName="r2"
).evaluate(baseline_df)

print("\n=== Baseline naive (prediction = Close actual) ===")
print(f"RMSE: {baseline_rmse:.6f}")
print(f"MAE : {baseline_mae:.6f}")
print(f"R²  : {baseline_r2:.6f}")


=== Modelo RandomForest ===
RMSE: 82.551699
MAE : 50.876139
R²  : 0.990213

=== Baseline naive (prediction = Close actual) ===
RMSE: 23.406329
MAE : 15.320032
R²  : 0.999213


In [ ]:

# Error absoluto y muestra de predicciones
result_df = (
    pred_df
    .select("Datetime", "Close", "next_close", "prediction")
    .withColumn("abs_error", F.abs(F.col("next_close") - F.col("prediction")))
)

result_df.orderBy("Datetime").show(20, truncate=False)


+-------------------+-------+----------+------------------+------------------+
|Datetime           |Close  |next_close|prediction        |abs_error         |
+-------------------+-------+----------+------------------+------------------+
|2024-09-26 11:00:00|2628.65|2630.2    |2666.37241532171  |36.172415321710105|
|2024-09-26 12:00:00|2630.2 |2620.63   |2666.8166570937465|46.18665709374636 |
|2024-09-26 13:00:00|2620.63|2613.59   |2667.243489003602 |53.65348900360186 |
|2024-09-26 14:00:00|2613.59|2652.15   |2669.6137896383525|17.46378963835241 |
|2024-09-26 15:00:00|2652.15|2650.17   |2680.9092507194864|30.739250719486336|
|2024-09-26 16:00:00|2650.17|2629.8    |2671.6707220070543|41.870722007054155|
|2024-09-26 17:00:00|2629.8 |2659.56   |2670.875943836145 |11.315943836144925|
|2024-09-26 18:00:00|2659.56|2649.43   |2679.7181009967767|30.288100996776848|
|2024-09-26 19:00:00|2649.43|2631.55   |2669.3691840622682|37.81918406226805 |
|2024-09-26 20:00:00|2631.55|2638.52   |2666.4044528


## 7. Importancia de variables

PySpark permite acceder a la importancia de las variables del Random Forest.


In [ ]:

rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray().tolist()

importance_rows = list(zip(feature_cols, importances))
importance_rows = sorted(importance_rows, key=lambda x: x[1], reverse=True)

for col_name, imp in importance_rows[:15]:
    print(f"{col_name:20s} -> {imp:.6f}")


Close                -> 0.307921
High                 -> 0.144594
sma_close_3          -> 0.135314
sma_close_6          -> 0.103867
Open                 -> 0.086060
close_lag_1          -> 0.077558
Low                  -> 0.048890
close_lag_4          -> 0.043104
close_lag_2          -> 0.042561
close_lag_3          -> 0.004960
sma_close_12         -> 0.004334
sma_volume_6         -> 0.000186
hl_ratio             -> 0.000085
std_close_12         -> 0.000078
amount_lag_1         -> 0.000068



## 8. Nuevas features
Añade al modelo nuevas variables que consideres.
Prueba a cambiar el target, predecir, por ejemplo, el cierre a 5 velas.


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Definir el nuevo TARGET: Cierre a 5 velas vista
w_order = Window.orderBy("Datetime")
df_nuevo_target = df.withColumn("next_close_5", F.lead("Close", 5).over(w_order))

# 2. Definir nuevas ventanas para las nuevas features
w_lag = Window.orderBy("Datetime")
w_ma_24 = Window.orderBy("Datetime").rowsBetween(-23, 0) # Ventana de las últimas 24 velas

# 3. Crear las nuevas variables
feat_df_5 = (
    df_nuevo_target
    # Variables básicas anteriores
    .withColumn("hour", F.hour("Datetime"))
    .withColumn("dayofweek", F.dayofweek("Datetime"))
    .withColumn("close_lag_1", F.lag("Close", 1).over(w_lag))
    .withColumn("return_1", F.when(F.lag("Close", 1).over(w_lag) != 0, (F.col("Close") - F.lag("Close", 1).over(w_lag)) / F.lag("Close", 1).over(w_lag)).otherwise(None))
    
    # NUEVAS VARIABLES
    .withColumn("return_5", F.when(F.lag("Close", 5).over(w_lag) != 0, (F.col("Close") - F.lag("Close", 5).over(w_lag)) / F.lag("Close", 5).over(w_lag)).otherwise(None)) # Retorno de las últimas 5 velas
    .withColumn("max_high_24", F.max("High").over(w_ma_24)) # Máximo de las últimas 24 velas
    .withColumn("min_low_24", F.min("Low").over(w_ma_24))   # Mínimo de las últimas 24 velas
    .withColumn("rango_24", F.col("max_high_24") - F.col("min_low_24")) # Amplitud del canal
).dropna()

print("Datos listos para modelar a 5 velas:", feat_df_5.count())

# 4. Split Temporal (80% Train, 20% Test)
feat_df_5 = feat_df_5.withColumn("row_id", F.row_number().over(w_order))
train_size_5 = int(feat_df_5.count() * 0.80)

train_df_5 = feat_df_5.filter(F.col("row_id") <= train_size_5)
test_df_5 = feat_df_5.filter(F.col("row_id") > train_size_5)

# 5. Pipeline de ML (Ensamblador + Random Forest)
nuevas_features_cols = [
    "Open", "High", "Low", "Close", "Volume", 
    "hour", "dayofweek", "close_lag_1", "return_1", 
    "return_5", "max_high_24", "min_low_24", "rango_24"
]

assembler_5 = VectorAssembler(inputCols=nuevas_features_cols, outputCol="features", handleInvalid="skip")
rf_5 = RandomForestRegressor(featuresCol="features", labelCol="next_close_5", predictionCol="prediction", numTrees=100, maxDepth=8, seed=42)

pipeline_5 = Pipeline(stages=[assembler_5, rf_5])

# Entrenar modelo
print("Entrenando modelo para predecir a 5 velas...")
modelo_5 = pipeline_5.fit(train_df_5)
pred_df_5 = modelo_5.transform(test_df_5)

# 6. Evaluación
evaluator_rmse_5 = RegressionEvaluator(labelCol="next_close_5", predictionCol="prediction", metricName="rmse")
evaluator_mae_5 = RegressionEvaluator(labelCol="next_close_5", predictionCol="prediction", metricName="mae")

rmse_5 = evaluator_rmse_5.evaluate(pred_df_5)
mae_5 = evaluator_mae_5.evaluate(pred_df_5)

print("\n=== Modelo RandomForest (Predicción a 5 velas) ===")
print(f"RMSE: {rmse_5:.6f}")
print(f"MAE : {mae_5:.6f}")

# Baseline naive para 5 velas (predecir que el precio dentro de 5 velas será exactamente el de hoy)
baseline_df_5 = test_df_5.withColumn("baseline_prediction", F.col("Close"))
baseline_rmse_5 = evaluator_rmse_5.setPredictionCol("baseline_prediction").evaluate(baseline_df_5)
baseline_mae_5 = evaluator_mae_5.setPredictionCol("baseline_prediction").evaluate(baseline_df_5)

print("\n=== Baseline naive (Prediction = Close actual) ===")
print(f"RMSE: {baseline_rmse_5:.6f}")
print(f"MAE : {baseline_mae_5:.6f}")

# Mostrar resultados
pred_df_5.select("Datetime", "Close", "next_close_5", "prediction").show(10, truncate=False)

